In [ ]:
from dataclasses import dataclass
import numpy as np
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, brier_score_loss, f1_score, log_loss,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import json
from datetime import datetime, timezone
from pathlib import Path
import joblib
import sklearn
import csv
from functools import lru_cache
from typing import Annotated
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field, FiniteFloat

import pytest
from fastapi.testclient import TestClient
from tempfile import TemporaryDirectory
import uvicorn


In [ ]:
PROJECT_DIR = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "cancer_var.ipynb").exists()
)
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
ARTIFACT_PATH = ARTIFACT_DIR / "breast_cancer_pipeline.joblib"
RUN_API = False


In [ ]:
RANDOM_STATE = 42


In [ ]:
def malignant_probability(model, rows):
    index = int(np.flatnonzero(model.classes_ == 0)[0])
    return model.predict_proba(rows)[:, index]


In [ ]:
def evaluate(target, scores):
    positive = np.asarray(target) == 0
    predicted = np.asarray(scores) >= 0.5
    return {
        "accuracy": float(accuracy_score(positive, predicted)),
        "precision_malignant": float(precision_score(positive, predicted, zero_division=0)),
        "recall_malignant": float(recall_score(positive, predicted, zero_division=0)),
        "f1_malignant": float(f1_score(positive, predicted, zero_division=0)),
        "brier_score": float(brier_score_loss(positive, scores)),
        "log_loss": float(log_loss(positive, scores, labels=[False, True])),
        "roc_auc_malignant": (
            float(roc_auc_score(positive, scores)) if len(np.unique(positive)) == 2 else None
        ),
    }


In [ ]:
@dataclass(frozen=True)
class TrainingResult:
    model: object
    feature_names: list[str]
    metrics: dict
    report: dict
    reference: dict


In [ ]:
def train_model():
    dataset = load_breast_cancer()
    train_ids, test_ids = train_test_split(
        np.arange(len(dataset.target)), test_size=0.2,
        random_state=RANDOM_STATE, stratify=dataset.target,
    )
    X_train, X_test = dataset.data[train_ids], dataset.data[test_ids]
    y_train, y_test = dataset.target[train_ids], dataset.target[test_ids]
    candidates = {
        "logistic_regression": make_pipeline(
            StandardScaler(), LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        ),
        "random_forest": RandomForestClassifier(
            n_estimators=100, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=1,
        ),
    }
    candidates.update({
        f"{name}_calibrated": CalibratedClassifierCV(
            estimator=model, method="sigmoid",
            cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
        )
        for name, model in list(candidates.items())
    })
    comparisons = {}
    for name, model in candidates.items():
        scores = cross_validate(
            model, X_train, y_train,
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
            scoring={"roc_auc": "roc_auc", "log_loss": "neg_log_loss"},
            error_score="raise", n_jobs=1,
        )
        comparisons[name] = {
            "roc_auc_mean": float(scores["test_roc_auc"].mean()),
            "roc_auc_std": float(scores["test_roc_auc"].std()),
            "log_loss_mean": float(-scores["test_log_loss"].mean()),
        }
    selected = min(comparisons, key=lambda name: comparisons[name]["log_loss_mean"])
    model = candidates[selected].fit(X_train, y_train)
    scores = malignant_probability(model, X_test)
    metrics = evaluate(y_test, scores)
    fraction, mean_probability = calibration_curve(y_test == 0, scores, n_bins=5)
    return TrainingResult(
        model=model, feature_names=list(dataset.feature_names), metrics=metrics,
        reference={"mean": X_train.mean(axis=0).tolist(), "std": X_train.std(axis=0).tolist()},
        report={
            "seed": RANDOM_STATE, "selected_model": selected,
            "selection_metric": "lowest_cv_log_loss", "cross_validation": comparisons,
            "test_metrics": metrics, "train_indices": train_ids.tolist(),
            "test_indices": test_ids.tolist(),
            "calibration_curve": {
                "mean_probability": mean_probability.tolist(),
                "fraction_malignant": fraction.tolist(),
            },
        },
    )


In [ ]:
result = train_model()
result.report["cross_validation"]


In [ ]:
result.metrics


In [ ]:
def save_result(result):
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    metadata = {
        "created_at": datetime.now(timezone.utc).isoformat(),
        "sklearn_version": sklearn.__version__,
        "selected_model": result.report["selected_model"],
    }
    joblib.dump({
        "model": result.model, "feature_names": result.feature_names,
        "reference": result.reference, "metadata": metadata,
        "test_metrics": result.metrics,
    }, ARTIFACT_DIR / "breast_cancer_pipeline.joblib")
    for filename, payload in (
        ("metrics.json", result.metrics),
        ("training_report.json", {**result.report, **metadata}),
    ):
        (ARTIFACT_DIR / filename).write_text(
            json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8",
        )
    print(json.dumps({**metadata, "metrics": result.metrics}, indent=2))


In [ ]:
save_result(result)


In [ ]:
def monitor(artifact, rows, target=None, shift_threshold=0.5):
    rows = np.asarray(rows, dtype=float)
    names = artifact["feature_names"]
    if rows.ndim != 2 or rows.shape[1] != len(names) or len(rows) == 0:
        raise ValueError("Expected a nonempty matrix matching feature_names")
    if not np.isfinite(rows).all():
        raise ValueError("Features must be finite")
    if not np.isfinite(shift_threshold) or shift_threshold <= 0:
        raise ValueError("shift_threshold must be positive and finite")
    reference = artifact["reference"]
    scale = np.maximum(np.asarray(reference["std"]), 1e-12)
    shifts = np.abs(rows.mean(axis=0) - np.asarray(reference["mean"])) / scale
    scores = malignant_probability(artifact["model"], rows)
    report = {
        "sample_count": len(rows),
        "shift_threshold": shift_threshold,
        "mean_probability_malignant": float(scores.mean()),
        "standardized_mean_shifts": dict(zip(names, shifts.tolist())),
        "shifted_features": [name for name, shift in zip(names, shifts) if shift > shift_threshold],
        "quality": None,
    }
    if target is not None:
        target = np.asarray(target)
        if target.shape != (len(rows),) or not np.isin(target, [0, 1]).all():
            raise ValueError("Targets must have one 0 or 1 label per row")
        report["quality"] = evaluate(target, scores)
        baseline = artifact.get("test_metrics", {})
        report["quality_delta_from_test"] = {
            key: value - baseline[key]
            for key, value in report["quality"].items()
            if value is not None and baseline.get(key) is not None
        }
    return report


In [ ]:
def monitor_csv(csv_path, output_path, model_path=ARTIFACT_PATH, shift_threshold=0.5):
    artifact = joblib.load(model_path)
    with Path(csv_path).open(encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        names = artifact["feature_names"]
        fields = reader.fieldnames or []
        if len(fields) != len(set(fields)) or not set(names).issubset(fields):
            raise ValueError("CSV must have unique headers including all feature names")
        records = list(reader)
    rows = [[float(row[name]) for name in names] for row in records]
    target = [float(row["target"]) for row in records] if "target" in fields else None
    report = monitor(artifact, rows, target, shift_threshold)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(report, indent=2, allow_nan=False), encoding="utf-8")
    return report


In [ ]:
dataset = load_breast_cancer()
artifact = joblib.load(ARTIFACT_PATH)
test_indices = result.report["test_indices"]
monitoring_report = monitor(artifact, dataset.data[test_indices], dataset.target[test_indices])
(ARTIFACT_DIR / "monitoring_report.json").write_text(
    json.dumps(monitoring_report, indent=2, allow_nan=False), encoding="utf-8"
)
monitoring_report


In [ ]:
app = FastAPI(
    title="Cancer Var — учебный API",
    description=(
        "Демонстрация на табличном наборе scikit-learn. "
        "Не предназначено для медицинской диагностики или принятия решений."
    ),
    version="0.1.0",
)

class PredictionRequest(BaseModel):
    features: Annotated[list[FiniteFloat], Field(min_length=30, max_length=30)]

@app.get("/health")
def health() -> dict[str, str]:
    if not ARTIFACT_PATH.exists():
        raise HTTPException(
            status_code=503,
            detail="Модель не найдена. Выполните ячейки обучения и сохранения модели",
        )
    return {"status": "ok", "model": "trained"}

@app.post("/predict")
def predict(request: PredictionRequest) -> dict[str, float | str]:
    if not ARTIFACT_PATH.exists():
        raise HTTPException(
            status_code=503,
            detail="Модель не найдена. Выполните ячейки обучения и сохранения модели",
        )

    artifact = load_artifact()
    model = artifact["model"]
    expected_features = len(artifact["feature_names"])
    if len(request.features) != expected_features:
        raise HTTPException(status_code=422, detail=f"Ожидается {expected_features} признаков")

    row = np.asarray(request.features, dtype=float).reshape(1, -1)
    if not np.isfinite(row).all():
        raise HTTPException(status_code=422, detail="Все признаки должны быть конечными числами")

    probabilities = model.predict_proba(row)[0]
    malignant_index = int(np.flatnonzero(model.classes_ == 0)[0])
    score = float(probabilities[malignant_index])
    return {
        "model_probability_malignant": score,
        "model_label": "malignant" if score >= 0.5 else "benign",
        "notice": "Учебная оценка модели; не является медицинским диагнозом или риском пациента.",
    }

@lru_cache(maxsize=1)
def load_artifact() -> dict:
    return joblib.load(ARTIFACT_PATH)

@app.get("/features")
def features() -> dict:
    health()
    artifact = load_artifact()
    return {"feature_names": artifact["feature_names"], "count": 30}


In [ ]:
assert not set(result.report["train_indices"]) & set(test_indices)
assert len(result.report["train_indices"]) + len(test_indices) == 569
comparisons = result.report["cross_validation"]
assert len(comparisons) == 4
assert result.report["selected_model"] == min(
    comparisons, key=lambda name: comparisons[name]["log_loss_mean"]
)
assert all(value >= 0 for value in result.metrics.values())
assert result.metrics["roc_auc_malignant"] > 0.9
load_artifact.cache_clear()
with TestClient(app) as client:
    row = dataset.data[0]
    response = client.post("/predict", json={"features": row.tolist()})
    assert response.status_code == 200
    assert response.json()["model_probability_malignant"] == pytest.approx(
        malignant_probability(result.model, [row])[0]
    )
    assert client.get("/features").json()["feature_names"] == result.feature_names
    for values in ([], [1.0]*29, [1.0]*31, ["invalid"]*30):
        assert client.post("/predict", json={"features": values}).status_code == 422
    original_path = ARTIFACT_PATH
    try:
        with TemporaryDirectory() as temporary:
            ARTIFACT_PATH = Path(temporary) / "missing.joblib"
            assert client.get("/health").status_code == 503
            assert client.post("/predict", json={"features": [1.0]*30}).status_code == 503
    finally:
        ARTIFACT_PATH = original_path
        load_artifact.cache_clear()
print("Model and API checks passed")


In [ ]:
class ConstantModel:
    classes_ = np.array([0, 1])

    def predict_proba(self, rows):
        return np.tile([0.8, 0.2], (len(rows), 1))


fixture_artifact = {
    "model": ConstantModel(), "feature_names": ["a", "b"],
    "reference": {"mean": [0, 0], "std": [1, 1]},
    "test_metrics": {"brier_score": 0.02},
}
shift_report = monitor(fixture_artifact, [[0, 3], [0, 3]])
assert shift_report["shifted_features"] == ["b"]
assert shift_report["quality"] is None
single_class_report = monitor(fixture_artifact, [[0, 0], [0, 0]], [0, 0])
assert single_class_report["quality"]["roc_auc_malignant"] is None
assert single_class_report["quality"]["brier_score"] == pytest.approx(0.04)
assert single_class_report["quality_delta_from_test"]["brier_score"] == pytest.approx(0.02)
for rows in ([], [[1]], [[float("nan"), 0]], [[float("inf"), 0]]):
    with pytest.raises(ValueError):
        monitor(fixture_artifact, rows)
with pytest.raises(ValueError):
    monitor(fixture_artifact, [[0, 0]], [2])
with TemporaryDirectory() as temporary:
    csv_path = Path(temporary) / "batch.csv"
    with csv_path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.writer(stream)
        writer.writerow([*result.feature_names, "target"])
        for index in test_indices[:10]:
            writer.writerow([*dataset.data[index], dataset.target[index]])
    csv_report = monitor_csv(csv_path, Path(temporary) / "report.json")
    assert csv_report["sample_count"] == 10
    assert csv_report["quality"] is not None
print("Monitoring and CSV checks passed")


In [ ]:
if RUN_API:
    server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8000))
    await server.serve()
